# LeetCode 2291 — Maximum Profit From Trading Stocks bằng Python

Link đề: https://leetcode.com/problems/maximum-profit-from-trading-stocks/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming

> **Bài Premium:** đề bài trong notebook được viết theo hiểu biết về bài (nguồn doocs). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc và chữ ký hàm.

Skeleton LeetCode:

```python
class Solution:
    def maximumProfit(self, present: List[int], future: List[int], budget: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra bài toán là **knapsack 0/1** (cái túi 0/1): mỗi cổ phiếu là một "món đồ" có **trọng lượng** `present[i]` (tiền bỏ ra) và **giá trị** `future[i] − present[i]` (lợi nhuận); sức chứa là `budget`.
2. Xây dựng DP 2 chiều `f[i][j]` rồi **nén** thành 1 chiều; hiểu vì sao vòng ngân sách phải chạy **ngược** (từ lớn về nhỏ).
3. Phân biệt "ngân sách **tối đa** `j`" (khởi tạo 0) với "tiêu **đúng** `j`" (khởi tạo `−∞`).
4. Thấy các heuristic tham lam (theo tỉ lệ lợi nhuận/giá, theo lợi nhuận lớn nhất) **sai**, bằng phản ví dụ tìm bằng vét cạn.
5. Truy vết lại **tập cổ phiếu được mua** từ bảng DP.

## 2. Đề bài & ví dụ

Cho hai mảng cùng độ dài `present` và `future`: `present[i]` là giá hiện tại của cổ phiếu `i`, `future[i]` là giá của nó sau một năm. Mỗi cổ phiếu được mua **tối đa một lần**. Cho `budget` là số tiền hiện có. Trả về **lợi nhuận lớn nhất** có thể đạt (lợi nhuận = tổng giá bán sau một năm − tổng tiền mua).

**Ví dụ 1:** `present = [5,4,6,2,3]`, `future = [8,5,4,3,5]`, `budget = 10` → `6` (mua 0, 3, 4: trả 10, bán 16).

**Ví dụ 2:** `present = [2,2,5]`, `future = [3,4,10]`, `budget = 6` → `5` (chỉ mua cổ phiếu 2).

**Ví dụ 3:** `present = [3,3,12]`, `future = [0,3,15]`, `budget = 10` → `0`.

**Ràng buộc:** `1 ≤ n ≤ 1000`; `0 ≤ present[i], future[i] ≤ 100`; `0 ≤ budget ≤ 1000`.

In [1]:
from typing import List
from functools import lru_cache
from itertools import combinations
import random, time

examples = [([5, 4, 6, 2, 3], [8, 5, 4, 3, 5], 10, 6), ([2, 2, 5], [3, 4, 10], 6, 5), ([3, 3, 12], [0, 3, 15], 10, 0)]
for p, f, b, exp in examples:
    print(f"present={p} future={f} budget={b} -> kỳ vọng {exp}; lợi nhuận từng cổ phiếu: {[y - x for x, y in zip(p, f)]}")

present=[5, 4, 6, 2, 3] future=[8, 5, 4, 3, 5] budget=10 -> kỳ vọng 6; lợi nhuận từng cổ phiếu: [3, 1, -2, 1, 2]
present=[2, 2, 5] future=[3, 4, 10] budget=6 -> kỳ vọng 5; lợi nhuận từng cổ phiếu: [1, 2, 5]
present=[3, 3, 12] future=[0, 3, 15] budget=10 -> kỳ vọng 0; lợi nhuận từng cổ phiếu: [-3, 0, 3]


## 3. Kiến thức nền

### 3.1 Knapsack 0/1

Có `n` món đồ, món `i` nặng `wᵢ`, giá trị `vᵢ`; chọn tập con có tổng trọng lượng ≤ `W` sao cho tổng giá trị lớn nhất. Mỗi món chọn **0 hoặc 1** lần. Bài toán NP-khó theo nghĩa tổng quát, nhưng giải được trong O(n·W) khi `W` nhỏ (thuật toán **giả đa thức**).

Ở bài này: `wᵢ = present[i]` (≤ 100), `vᵢ = future[i] − present[i]`, `W = budget` (≤ 1000) ⇒ `n · W ≤ 10⁶`.

In [2]:
items = [(p, f - p) for p, f in zip(*examples[0][:2])]
print("(trọng lượng, giá trị):", items)
print("chỉ những món có giá trị dương mới đáng cân nhắc:", [it for it in items if it[1] > 0])

(trọng lượng, giá trị): [(5, 3), (4, 1), (6, -2), (2, 1), (3, 2)]
chỉ những món có giá trị dương mới đáng cân nhắc: [(5, 3), (4, 1), (2, 1), (3, 2)]


### 3.2 Món đồ giá trị âm hoặc bằng 0 thì không bao giờ cần mua

Bỏ một món có `vᵢ ≤ 0` khỏi một tập hợp lệ: tổng trọng lượng không tăng (vẫn hợp lệ, vì `wᵢ ≥ 0`), tổng giá trị không giảm. ⇒ Có thể lọc trước, chỉ giữ các cổ phiếu **lời**.

### 3.3 Công thức truy hồi

`f[i][j]` = lợi nhuận lớn nhất khi chỉ xét `i` cổ phiếu đầu, tiêu **tối đa** `j`:

- không mua cổ phiếu `i`: `f[i−1][j]`;
- mua (nếu `j ≥ wᵢ`): `f[i−1][j − wᵢ] + vᵢ`.

`f[i][j] = max` của hai lựa chọn; `f[0][j] = 0`.

### 3.4 Vì sao 1 chiều phải duyệt `j` **giảm dần**

Khi tính `f[j]` mới, ta cần `f[j − w]` của **hàng cũ**. Duyệt `j` giảm dần ⇒ ô `j − w < j` chưa bị ghi đè ở vòng này ⇒ vẫn là giá trị hàng cũ. Duyệt tăng dần thì `f[j − w]` đã được cập nhật (có thể đã chứa món `i`) ⇒ món `i` bị dùng **nhiều lần** — đó là knapsack **không giới hạn**.

In [3]:
def knap_1d(items, W, descending=True):
    f = [0] * (W + 1)
    for w, v in items:
        rng_j = range(W, w - 1, -1) if descending else range(w, W + 1)
        for j in rng_j:
            f[j] = max(f[j], f[j - w] + v)
    return f[W]

print("1 món (w=2, v=3), W=6:  giảm dần =", knap_1d([(2, 3)], 6), "| tăng dần =", knap_1d([(2, 3)], 6, False), "(dùng 3 lần!)")

1 món (w=2, v=3), W=6:  giảm dần = 3 | tăng dần = 9 (dùng 3 lần!)


## 4. Ý tưởng nền tảng / trực giác

Mỗi cổ phiếu là một quyết định nhị phân "mua / không mua", ràng buộc duy nhất là **tổng tiền mua ≤ budget**. Đây đúng là knapsack 0/1. Ta không thể tham lam vì một cổ phiếu "rất lời" nhưng đắt có thể chiếm chỗ của vài cổ phiếu rẻ cộng lại còn lời hơn.

Tính tay ví dụ 1 sau khi lọc cổ phiếu lời: `(w, v) = (5, 3), (4, 1), (2, 1), (3, 2)` (cổ phiếu 2 lỗ: bỏ). Ngân sách 10.

| tập | tổng giá | lợi nhuận |
|---|---|---|
| {0, 1} | 9 | 4 |
| {0, 3, 4} | 10 | **6** |
| {0, 1} + bất kỳ | > 10 | — |
| {1, 3, 4} | 9 | 4 |
| {0, 4} | 8 | 5 |

⇒ 6 ✔. Ví dụ 3: cổ phiếu duy nhất lời là `(12, 3)` nhưng giá 12 > 10 ⇒ 0.

In [4]:
p, f, b, _ = examples[0]
best = max(((sum(p[i] for i in S), sum(f[i] - p[i] for i in S), S)
            for r in range(len(p) + 1) for S in combinations(range(len(p)), r) if sum(p[i] for i in S) <= b),
           key=lambda t: t[1])
print("tốt nhất: tập", best[2], "tổng giá", best[0], "lợi nhuận", best[1])

tốt nhất: tập (0, 3, 4) tổng giá 10 lợi nhuận 6


## 5. Cách 1 — Brute Force / baseline

### 5.1 Duyệt mọi tập con — O(2ⁿ · n)

In [5]:
def brute(present, future, budget):
    n = len(present); best = 0
    for mask in range(1 << n):
        cost = gain = 0
        for i in range(n):
            if mask >> i & 1:
                cost += present[i]; gain += future[i] - present[i]
        if cost <= budget: best = max(best, gain)
    return best

for p, f, b, exp in examples:
    print(brute(p, f, b), "OK" if brute(p, f, b) == exp else "SAI")

6 OK
5 OK
0 OK


**Độ phức tạp:** `2¹⁰⁰⁰` — bất khả thi. Chỉ dùng kiểm chứng với `n ≤ 14`.

### 5.2 Baseline 2 — Đệ quy "mua / không mua" có nhớ — O(n · budget) trạng thái

In [6]:
def memo(present, future, budget):
    n = len(present)
    @lru_cache(maxsize=None)
    def go(i, left):
        if i == n: return 0
        best = go(i + 1, left)
        if present[i] <= left:
            best = max(best, future[i] - present[i] + go(i + 1, left - present[i]))
        return best
    r = go(0, budget); go.cache_clear()
    return r

for p, f, b, exp in examples:
    print(memo(p, f, b), "OK" if memo(p, f, b) == exp else "SAI")

6 OK
5 OK
0 OK


Đệ quy có nhớ đúng, nhưng với `n = 1000` độ sâu đệ quy 1000 sát giới hạn mặc định của Python, và `lru_cache` với tới `10⁶` trạng thái tốn bộ nhớ và chậm. **Nút thắt:** chuyển sang **lặp từ dưới lên** (bottom-up), rồi nén bộ nhớ.

## 6. Ý tưởng thuật toán / bước đột phá

**DP 1 chiều.** `f[j]` = lợi nhuận lớn nhất với ngân sách tối đa `j` khi đã xét một số cổ phiếu đầu.

```
f = [0] * (budget + 1)
for mỗi cổ phiếu (w, v) với v > 0:
    for j từ budget xuống w:
        f[j] = max(f[j], f[j - w] + v)
return f[budget]
```

- **Khởi tạo 0** cho mọi `j`: tập rỗng hợp lệ với mọi ngân sách, lợi nhuận 0. Nhờ vậy `f[j]` mang nghĩa "**tối đa** `j`" và `f` không giảm theo `j`.
- **Duyệt `j` giảm dần**: đảm bảo mỗi cổ phiếu dùng tối đa một lần.
- **Lọc `v > 0`**: không đổi kết quả, giảm số món.

Một cách viết khác **không cần nghĩ tới chiều duyệt**: tạo hàng mới từ hàng cũ bằng list comprehension: `g = f[:w] + [max(f[j], f[j - w] + v) for j in range(w, W + 1)]` — vì mọi ô của `g` đọc từ `f` cũ.

**Độ phức tạp:** O(n · budget) ≤ `10⁶` phép tính; bộ nhớ O(budget).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DP 2 chiều (cách doocs 1)

In [7]:
def dp2d(present, future, budget):
    n = len(present)
    f = [[0] * (budget + 1) for _ in range(n + 1)]
    for i, w in enumerate(present, 1):
        for j in range(budget + 1):
            f[i][j] = f[i - 1][j]
            if j >= w and future[i - 1] > w:
                f[i][j] = max(f[i][j], f[i - 1][j - w] + future[i - 1] - w)
    return f[n][budget]

for p, f, b, exp in examples:
    print(dp2d(p, f, b), "OK" if dp2d(p, f, b) == exp else "SAI")

6 OK
5 OK
0 OK


### 7.2 Cách 3 — DP 1 chiều duyệt ngược (cách doocs 2)

In [8]:
def dp1d(present, future, budget):
    f = [0] * (budget + 1)
    for a, b in zip(present, future):
        if b <= a: continue                       # không lời: bỏ qua
        for j in range(budget, a - 1, -1):
            f[j] = max(f[j], f[j - a] + b - a)
    return f[budget]

for p, f, b, exp in examples:
    print(dp1d(p, f, b), "OK" if dp1d(p, f, b) == exp else "SAI")

6 OK
5 OK
0 OK


### 7.3 Cách 4 — Hàng mới bằng list comprehension (nhanh hơn trong Python)

In [9]:
def dp_rows(present, future, budget):
    f = [0] * (budget + 1)
    for a, b in zip(present, future):
        v = b - a
        if v <= 0 or a > budget: continue
        f = f[:a] + [x if x >= y + v else y + v for x, y in zip(f[a:], f)]
    return f[budget]

for p, f, b, exp in examples:
    print(dp_rows(p, f, b), "OK" if dp_rows(p, f, b) == exp else "SAI")

6 OK
5 OK
0 OK


`zip(f[a:], f)` ghép `f[j]` với `f[j − a]` cho `j = a … budget` (vì `zip` dừng ở dãy ngắn hơn). Mọi giá trị đều đọc từ `f` **cũ** ⇒ đúng semantics 0/1 mà không cần duyệt ngược. Vòng lặp chạy trong C nhiều hơn ⇒ nhanh hơn 2–4 lần.

### 7.4 Cách 5 — DP "theo lợi nhuận": chi phí nhỏ nhất để đạt lợi nhuận `p`

Đổi vai trò: `g[p]` = tổng tiền mua **nhỏ nhất** để có lợi nhuận **đúng** `p`. Đáp án = `max p` với `g[p] ≤ budget`. Lợi nhuận tối đa ≤ `100 · n`, nên cách này chỉ đáng dùng khi **ngân sách lớn** mà lợi nhuận nhỏ (mục 15.3).

In [10]:
def dp_by_profit(present, future, budget):
    items = [(a, b - a) for a, b in zip(present, future) if b > a]
    P = sum(v for _, v in items); INF = float("inf")
    g = [0] + [INF] * P
    for w, v in items:
        for p in range(P, v - 1, -1):
            if g[p - v] + w < g[p]: g[p] = g[p - v] + w
    return max(p for p in range(P + 1) if g[p] <= budget)

for p, f, b, exp in examples:
    print(dp_by_profit(p, f, b), "OK" if dp_by_profit(p, f, b) == exp else "SAI")

6 OK
5 OK
0 OK


## 8. TRACE từng bước

### 8.1 Trace DP 1 chiều trên ví dụ 1 (in mảng `f` sau mỗi cổ phiếu)

In [11]:
def trace(present, future, budget):
    f = [0] * (budget + 1)
    print("ngân sách j :", list(range(budget + 1)))
    print("ban đầu     :", f)
    for i, (a, b) in enumerate(zip(present, future)):
        if b <= a:
            print(f"cổ phiếu {i} (giá {a}, lời {b - a:+d}) -> bỏ qua"); continue
        for j in range(budget, a - 1, -1):
            f[j] = max(f[j], f[j - a] + b - a)
        print(f"cổ phiếu {i} (giá {a}, lời {b - a:+d}):", f)
    return f[budget]

print("đáp án:", trace(*examples[0][:3]))

ngân sách j : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
ban đầu     : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
cổ phiếu 0 (giá 5, lời +3): [0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3]
cổ phiếu 1 (giá 4, lời +1): [0, 0, 0, 0, 1, 3, 3, 3, 3, 4, 4]
cổ phiếu 2 (giá 6, lời -2) -> bỏ qua
cổ phiếu 3 (giá 2, lời +1): [0, 0, 1, 1, 1, 3, 3, 4, 4, 4, 4]
cổ phiếu 4 (giá 3, lời +2): [0, 0, 1, 2, 2, 3, 3, 4, 5, 5, 6]
đáp án: 6


Đọc cột `j = 10` ở dòng cuối: `6`. Ở dòng sau cổ phiếu 3 (giá 2, lời 1), `f[7] = f[5] + 1 = 3 + 1 = 4` (cổ phiếu 0 và 3, tổng giá 7). Sau cổ phiếu 4 (giá 3, lời 2): `f[10] = f[7] + 2 = 6` — đúng tập {0, 3, 4}. Chú ý `f[9]` sau cổ phiếu 1 là `4` (cổ phiếu 0 + 1, giá 9) nhưng không "mở rộng" được thêm trong ngân sách 10.

### 8.2 Truy vết tập cổ phiếu được mua (cần bảng 2 chiều)

In [12]:
def best_set(present, future, budget):
    n = len(present)
    f = [[0] * (budget + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        w, v = present[i - 1], future[i - 1] - present[i - 1]
        for j in range(budget + 1):
            f[i][j] = f[i - 1][j]
            if j >= w and v > 0 and f[i - 1][j - w] + v > f[i][j]:
                f[i][j] = f[i - 1][j - w] + v
    chosen = []; j = budget
    for i in range(n, 0, -1):
        if f[i][j] != f[i - 1][j]:              # giá trị thay đổi => đã mua cổ phiếu i-1
            chosen.append(i - 1); j -= present[i - 1]
    return f[n][budget], sorted(chosen)

for p, f, b, exp in examples:
    val, S = best_set(p, f, b)
    print(f"lợi nhuận {val}, mua {S}, tổng giá {sum(p[i] for i in S)} <= {b}, kiểm tra lời {sum(f[i] - p[i] for i in S)}")

lợi nhuận 6, mua [0, 3, 4], tổng giá 10 <= 10, kiểm tra lời 6
lợi nhuận 5, mua [2], tổng giá 5 <= 6, kiểm tra lời 5
lợi nhuận 0, mua [], tổng giá 0 <= 10, kiểm tra lời 0


Truy vết đi **ngược** từ `(n, budget)`: nếu `f[i][j] ≠ f[i−1][j]` thì bắt buộc đã mua cổ phiếu `i−1` (lựa chọn "không mua" không đạt được giá trị này), giảm `j` đi giá của nó. DP 1 chiều làm mất thông tin này — muốn truy vết cần bảng 2 chiều (hoặc lưu bit "đã mua" cho từng `(i, j)`).

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def maximumProfit(self, present: List[int], future: List[int], budget: int) -> int:
        f = [0] * (budget + 1)                    # f[j]: lời lớn nhất với ngân sách tối đa j
        for a, b in zip(present, future):
            v = b - a
            if v <= 0:                            # cổ phiếu không lời: không bao giờ cần mua
                continue
            for j in range(budget, a - 1, -1):    # duyệt ngược: mỗi cổ phiếu mua tối đa một lần
                if f[j - a] + v > f[j]:
                    f[j] = f[j - a] + v
        return f[budget]

In [14]:
sol = Solution()
for p, f, b, exp in examples:
    got = sol.maximumProfit(p, f, b)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

6 | kỳ vọng 6 | OK
5 | kỳ vọng 5 | OK
0 | kỳ vọng 0 | OK


## 10. Vì sao thuật toán đúng

**Định nghĩa.** `F(i, j)` = max lợi nhuận trên các tập con `S ⊆ {0, …, i−1}` với `Σ_{k∈S} present[k] ≤ j`.

**Truy hồi.** Một tập tối ưu `S` cho `(i, j)` hoặc không chứa `i−1` — khi đó `S` là tập hợp lệ cho `(i−1, j)`, giá trị ≤ `F(i−1, j)`; hoặc chứa `i−1` (cần `j ≥ w`) — khi đó `S \ {i−1}` hợp lệ cho `(i−1, j − w)`, giá trị ≤ `F(i−1, j−w) + v`. Ngược lại, mọi giá trị ở hai vế đều đạt được bằng cách thêm/không thêm `i−1` vào tập tối ưu tương ứng. ⇒ `F(i, j) = max(F(i−1, j), F(i−1, j−w) + v [nếu j ≥ w])`, `F(0, j) = 0`.

**Nén 1 chiều.** Bất biến: trước khi xử lý cổ phiếu `i−1`, `f[j] = F(i−1, j)` với mọi `j`. Trong vòng duyệt `j` giảm dần, khi cập nhật `f[j]`, ô `f[j − w]` (với `j − w < j`) **chưa** được cập nhật ở vòng này (và `w = 0` thì `j − w = j` là chính ô đang đọc trước khi ghi) ⇒ vẫn bằng `F(i−1, j−w)`. Vậy sau vòng, `f[j] = F(i, j)`. Ô `j < w` không đổi, đúng vì khi đó không mua được.

**Lọc `v ≤ 0`.** Với `v ≤ 0`, `F(i−1, j−w) + v ≤ F(i−1, j−w) ≤ F(i−1, j)` (vì `F` không giảm theo `j`) ⇒ nhánh "mua" không bao giờ thắng ⇒ bỏ qua không đổi kết quả.

**Kết luận:** `f[budget] = F(n, budget)`. ∎

## 11. Phân tích Time / Space Complexity

`n ≤ 1000`, `B = budget ≤ 1000`.

| Cách | Time | Space |
|---|---|---|
| 1. Tập con | O(2ⁿ · n) | O(1) |
| 1b. Đệ quy có nhớ | O(n · B) | O(n · B) + ngăn xếp đệ quy |
| 2. DP 2 chiều | O(n · B) | O(n · B) |
| 3. DP 1 chiều | O(n · B) | O(B) |
| 4. Hàng mới (list comprehension) | O(n · B) | O(B) |
| 5. Theo lợi nhuận | O(n · P), P ≤ 100n | O(P) |

In [15]:
rng = random.Random(2291)
n, B = 1000, 1000
P = [rng.randint(0, 100) for _ in range(n)]; Fu = [rng.randint(0, 100) for _ in range(n)]
res = []
for name, fn in [("dp2d", dp2d), ("dp1d", dp1d), ("dp_rows", dp_rows), ("Solution", Solution().maximumProfit)]:
    t0 = time.perf_counter(); res.append(fn(P, Fu, B)); dt = time.perf_counter() - t0
    print(f"{name:<9} {dt*1000:7.0f} ms -> {res[-1]}")
assert len(set(res)) == 1
t0 = time.perf_counter(); r5 = dp_by_profit(P[:200], Fu[:200], B); print(f"dp_by_profit (n=200): {(time.perf_counter()-t0)*1000:.0f} ms ->", r5, "| dp1d:", dp1d(P[:200], Fu[:200], B))

dp2d          126 ms -> 7897


dp1d           32 ms -> 7897


dp_rows        20 ms -> 7897


Solution       25 ms -> 7897


dp_by_profit (n=200): 19 ms -> 3125 | dp1d: 3125


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng — đối chiếu lại đề gốc nếu có Premium)

In [16]:
ALL = {"brute": brute, "memo": memo, "dp2d": dp2d, "dp1d": dp1d, "dp_rows": dp_rows,
       "dp_by_profit": dp_by_profit, "Solution": Solution().maximumProfit}
edge = examples + [
    ([1], [5], 0, 0),                     # ngân sách 0, cổ phiếu giá 1
    ([0], [5], 0, 5),                     # cổ phiếu miễn phí: mua được với ngân sách 0
    ([0, 0, 0], [1, 2, 0], 0, 3),         # nhiều cổ phiếu giá 0
    ([5, 5], [1, 2], 100, 0),             # toàn lỗ
    ([10], [20], 10, 10),                 # vừa khít ngân sách
    ([10], [20], 9, 0),                   # thiếu 1 đồng
    ([6, 3, 3], [12, 6, 6], 6, 6),        # 1 món đắt = 2 món rẻ: hòa
    ([4, 3, 3], [9, 6, 6], 6, 6),         # tham lam theo lời lớn nhất (5) thua 2 món rẻ (3+3)
    ([100] * 10, [100] * 10, 1000, 0),    # lời 0
]
for p, f, b, exp in edge:
    for name, fn in ALL.items():
        got = fn(p, f, b)
        assert got == exp, (name, p, f, b, got, exp)
    print(f"present={p!s:<20} future={f!s:<20} budget={b:<4} -> {exp}")
print("edge OK")

present=[5, 4, 6, 2, 3]      future=[8, 5, 4, 3, 5]      budget=10   -> 6
present=[2, 2, 5]            future=[3, 4, 10]           budget=6    -> 5
present=[3, 3, 12]           future=[0, 3, 15]           budget=10   -> 0
present=[1]                  future=[5]                  budget=0    -> 0
present=[0]                  future=[5]                  budget=0    -> 5
present=[0, 0, 0]            future=[1, 2, 0]            budget=0    -> 3
present=[5, 5]               future=[1, 2]               budget=100  -> 0
present=[10]                 future=[20]                 budget=10   -> 10
present=[10]                 future=[20]                 budget=9    -> 0
present=[6, 3, 3]            future=[12, 6, 6]           budget=6    -> 6
present=[4, 3, 3]            future=[9, 6, 6]            budget=6    -> 6
present=[100, 100, 100, 100, 100, 100, 100, 100, 100, 100] future=[100, 100, 100, 100, 100, 100, 100, 100, 100, 100] budget=1000 -> 0
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [17]:
rng = random.Random(42)
for _ in range(600):
    n = rng.randint(1, 10)
    p = [rng.randint(0, 12) for _ in range(n)]; f = [rng.randint(0, 15) for _ in range(n)]
    b = rng.randint(0, 40)
    ref = brute(p, f, b)
    for name, fn in ALL.items():
        assert fn(p, f, b) == ref, (name, p, f, b)
print("600 bộ ngẫu nhiên OK")

600 bộ ngẫu nhiên OK


### 12.3 Tính chất: không giảm theo ngân sách; ngân sách đủ lớn ⇒ bằng tổng lời dương; hoán vị cổ phiếu không đổi đáp án

In [18]:
rng = random.Random(7)
for _ in range(300):
    n = rng.randint(1, 12)
    p = [rng.randint(0, 20) for _ in range(n)]; f = [rng.randint(0, 25) for _ in range(n)]
    vals = [Solution().maximumProfit(p, f, b) for b in range(0, 60)]
    assert all(x <= y for x, y in zip(vals, vals[1:]))
    assert Solution().maximumProfit(p, f, sum(p)) == sum(max(0, y - x) for x, y in zip(p, f))
    idx = list(range(n)); rng.shuffle(idx)
    assert Solution().maximumProfit([p[i] for i in idx], [f[i] for i in idx], 30) == vals[30]
print("tính chất OK")

tính chất OK


### 12.4 Edge cực trị: n = 1000 cổ phiếu giá 1 lời 99, ngân sách 1000 ⇒ 99000; n = 1000 cổ phiếu giá 0 ⇒ 100000 dù ngân sách 0

In [19]:
print(Solution().maximumProfit([1] * 1000, [100] * 1000, 1000), "(giá 1, bán 100: lời 99 mỗi cổ phiếu, mua cả 1000 -> 99000)")
assert Solution().maximumProfit([1] * 1000, [100] * 1000, 1000) == 99000
assert Solution().maximumProfit([0] * 1000, [100] * 1000, 0) == 100000
assert Solution().maximumProfit([100] * 1000, [100] * 1000, 1000) == 0
print("cực trị OK")

99000 (giá 1, bán 100: lời 99 mỗi cổ phiếu, mua cả 1000 -> 99000)
cực trị OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — DP 1 chiều duyệt `j` **tăng dần** (vô tình thành knapsack không giới hạn)

In [20]:
def wrong_forward(present, future, budget):
    f = [0] * (budget + 1)
    for a, b in zip(present, future):
        for j in range(a, budget + 1):                    # SAI: f[j - a] có thể đã chứa cổ phiếu này
            f[j] = max(f[j], f[j - a] + b - a)
    return f[budget]

for p, f, b, exp in examples:
    print("bản sai :", wrong_forward(p, f, b), "| bản đúng:", Solution().maximumProfit(p, f, b))

bản sai : 6 | bản đúng: 6
bản sai : 6 | bản đúng: 5
bản sai : 0 | bản đúng: 0


**Vì sao sai:** ví dụ 2: cổ phiếu giá 2 lời 2 bị "mua" 3 lần với ngân sách 6 ⇒ 6 > 5. **Sửa:** duyệt `j` từ `budget` xuống `a`.

### Lỗi 2 — Tham lam theo **tỉ lệ lợi nhuận / giá**

In [21]:
def wrong_ratio(present, future, budget):
    order = sorted(range(len(present)), key=lambda i: -(future[i] - present[i]) / present[i] if present[i] else -1e18)
    gain = 0; left = budget
    for i in order:
        if future[i] > present[i] and present[i] <= left:
            left -= present[i]; gain += future[i] - present[i]
    return gain

for p, f, b, exp in examples:
    print("bản sai :", wrong_ratio(p, f, b), "| bản đúng:", Solution().maximumProfit(p, f, b))

bản sai : 6 | bản đúng: 6
bản sai : 3 | bản đúng: 5
bản sai : 0 | bản đúng: 0


**Vì sao sai:** tham lam theo tỉ lệ chỉ đúng với knapsack **phân số** (được mua một phần). Ví dụ 1: thứ tự tỉ lệ là cổ phiếu 4 (2/3), 0 (3/5), 3 (1/2), 1 (1/4): mua 4, 0, 3 với tổng giá 10, lời 6 — **trùng hợp** đúng. Ví dụ 2: tỉ lệ là `0.5, 1.0, 1.0`; cổ phiếu 1 và 2 hòa, sắp xếp ổn định đặt cổ phiếu 1 trước ⇒ mua cổ phiếu 1 (còn 4 đồng), cổ phiếu 2 giá 5 không mua nổi, mua thêm cổ phiếu 0 ⇒ lời `2 + 1 = 3 < 5`.

### Lỗi 3 — Tham lam theo **lợi nhuận lớn nhất**

In [22]:
def wrong_max_profit(present, future, budget):
    order = sorted(range(len(present)), key=lambda i: present[i] - future[i])
    gain = 0; left = budget
    for i in order:
        if future[i] > present[i] and present[i] <= left:
            left -= present[i]; gain += future[i] - present[i]
    return gain

p, f, b = [4, 3, 3], [9, 6, 6], 6
print("bản sai :", wrong_max_profit(p, f, b), "| bản đúng:", Solution().maximumProfit(p, f, b))
print("ví dụ đề (trùng hợp đúng):", [wrong_max_profit(*e[:3]) for e in examples], [e[3] for e in examples])

bản sai : 5 | bản đúng: 6
ví dụ đề (trùng hợp đúng): [6, 5, 0] [6, 5, 0]


**Vì sao sai:** mua cổ phiếu lời 5 (giá 4) thì chỉ còn 2 đồng, không mua thêm được; hai cổ phiếu giá 3 lời 3 + 3 = 6 tốt hơn. Cả ba ví dụ của đề đều **không** phát hiện được lỗi này.

### Lỗi 4 — Khởi tạo `−∞` (nghĩa "tiêu **đúng** `j`") nhưng vẫn trả `f[budget]`

In [23]:
def wrong_exact(present, future, budget):
    NEG = float("-inf")
    f = [0] + [NEG] * budget                              # f[j]: tiêu ĐÚNG j
    for a, b in zip(present, future):
        for j in range(budget, a - 1, -1):
            f[j] = max(f[j], f[j - a] + b - a)
    return f[budget]                                      # SAI: phải là max(f)

for p, f, b, exp in examples:
    print("bản sai :", wrong_exact(p, f, b), "| bản đúng:", Solution().maximumProfit(p, f, b))

bản sai : 6 | bản đúng: 6
bản sai : -inf | bản đúng: 5
bản sai : -inf | bản đúng: 0


**Vì sao sai:** với khởi tạo `−∞`, `f[j]` là lời lớn nhất của các tập tiêu **đúng** `j`. Ví dụ 2: các tổng giá có thể là `0, 2, 4, 5, 7, 9` — không có tập nào tiêu đúng 6 ⇒ `f[6] = −∞`. Ví dụ 1 **trùng hợp** đúng vì tập tối ưu tiêu đúng 10. **Sửa:** khởi tạo 0 cho mọi `j` (nghĩa "tối đa `j`"), hoặc giữ `−∞` và trả `max(f)`.

### Lỗi 5 — DP 2 chiều quên sao chép hàng trên khi `j < w`

In [24]:
def wrong_no_copy(present, future, budget):
    n = len(present); f = [[0] * (budget + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        w, v = present[i - 1], future[i - 1] - present[i - 1]
        for j in range(w, budget + 1):                    # SAI: các ô j < w giữ 0 thay vì f[i-1][j]
            f[i][j] = max(f[i - 1][j], f[i - 1][j - w] + v)
    return f[n][budget]

p, f, b = [1, 5], [3, 6], 4
print("bản sai :", wrong_no_copy(p, f, b), "| bản đúng:", Solution().maximumProfit(p, f, b))

bản sai : 0 | bản đúng: 2


**Vì sao sai:** cổ phiếu thứ hai giá 5 > 4 nên hàng `i = 2` chỉ được điền từ `j = 5` trở đi; `f[2][4]` vẫn là 0 thay vì 2. **Sửa:** luôn gán `f[i][j] = f[i−1][j]` trước, rồi mới xét nhánh "mua".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chọn tập con, mỗi phần tử tối đa một lần, tổng chi phí ≤ C, tối đa tổng giá trị" ⇒ **knapsack 0/1**: O(n · C).
- 1 chiều: **0/1 ⇒ duyệt ngược**, **không giới hạn ⇒ duyệt xuôi**.
- Khởi tạo 0 ⇒ "tối đa C"; khởi tạo `−∞` (trừ ô 0) ⇒ "đúng C".
- Lọc món giá trị ≤ 0 trước cho nhanh (và an toàn).
- Chi phí lớn nhưng giá trị nhỏ ⇒ đổi chiều DP theo giá trị (mục 7.4).
- Tham lam theo tỉ lệ **chỉ** đúng cho knapsack phân số.
- Bài cùng họ: 416 Partition Equal Subset Sum, 494 Target Sum, 474 Ones and Zeroes, 1049 Last Stone Weight II, 879 Profitable Schemes, 2915 Length of the Longest Subsequence That Sums to Target.

## 15. Biến thể của bài

### 15.1 Biến thể A — Mỗi cổ phiếu mua **không giới hạn** số lần (knapsack không giới hạn)

In [25]:
def unbounded(present, future, budget):
    f = [0] * (budget + 1)
    for a, b in zip(present, future):
        if b <= a or a == 0: continue                     # không lời; giá 0 mà lời dương => lời vô hạn, loại khỏi biến thể
        for j in range(a, budget + 1):                    # duyệt xuôi: được dùng lại
            f[j] = max(f[j], f[j - a] + b - a)
    return f[budget]

def unbounded_brute(present, future, budget):
    @lru_cache(maxsize=None)
    def go(left):
        return max([0] + [future[i] - present[i] + go(left - present[i]) for i in range(len(present))
                          if 0 < present[i] <= left and future[i] > present[i]])
    return go(budget)

rng = random.Random(151)
for _ in range(300):
    n = rng.randint(1, 5); p = [rng.randint(1, 8) for _ in range(n)]; f = [rng.randint(0, 12) for _ in range(n)]
    b = rng.randint(0, 30)
    assert unbounded(p, f, b) == unbounded_brute(p, f, b)
print(unbounded(*examples[1][:3]), "(ví dụ 2 nếu mua lặp được)")

6 (ví dụ 2 nếu mua lặp được)


### 15.2 Biến thể B — Giới hạn thêm **số lượng** cổ phiếu được mua (tối đa `k` mã): DP 2 ràng buộc

In [26]:
def with_count_limit(present, future, budget, k):
    NEG = float("-inf")
    f = [[0] * (budget + 1) for _ in range(k + 1)]       # f[c][j]: mua tối đa c mã, tiêu tối đa j
    for a, b in zip(present, future):
        v = b - a
        if v <= 0: continue
        for c in range(k, 0, -1):
            row, prev = f[c], f[c - 1]
            for j in range(budget, a - 1, -1):
                if prev[j - a] + v > row[j]: row[j] = prev[j - a] + v
    return f[k][budget]

def with_count_brute(present, future, budget, k):
    n = len(present); best = 0
    for r in range(min(k, n) + 1):
        for S in combinations(range(n), r):
            if sum(present[i] for i in S) <= budget:
                best = max(best, sum(future[i] - present[i] for i in S))
    return best

rng = random.Random(152)
for _ in range(300):
    n = rng.randint(1, 8); p = [rng.randint(0, 10) for _ in range(n)]; f = [rng.randint(0, 15) for _ in range(n)]
    b = rng.randint(0, 30); k = rng.randint(0, n)
    assert with_count_limit(p, f, b, k) == with_count_brute(p, f, b, k)
print(with_count_limit(*examples[0][:3], 2), "(ví dụ 1 nếu chỉ được mua tối đa 2 mã)")

5 (ví dụ 1 nếu chỉ được mua tối đa 2 mã)


### 15.3 Biến thể C — Ngân sách cực lớn (ví dụ `10⁹`), giá lớn, nhưng lợi nhuận nhỏ

DP theo ngân sách không khả thi; dùng **DP theo lợi nhuận** (mục 7.4): O(n · Σ lời).

In [27]:
rng = random.Random(153)
p = [rng.randint(10**6, 10**8) for _ in range(40)]; f = [x + rng.randint(-5, 10) for x in p]
B = 10**9
t0 = time.perf_counter(); r = dp_by_profit(p, f, B); dt = time.perf_counter() - t0
print("lợi nhuận tối đa:", r, f"({dt*1000:.0f} ms) — DP theo ngân sách sẽ cần mảng 10^9 ô")
small = [(x, y) for x, y in zip(p, f)][:14]
assert r >= 0 and dp_by_profit([a for a, _ in small], [b for _, b in small], B) == brute([a for a, _ in small], [b for _, b in small], B)

lợi nhuận tối đa: 138 (0 ms) — DP theo ngân sách sẽ cần mảng 10^9 ô


### 15.4 Biến thể D — Tối đa **giá trị bán ra** thay vì lợi nhuận

Chỉ đổi giá trị món: `vᵢ = future[i]`. Lúc này cổ phiếu lỗ vẫn có thể đáng mua (nếu dư tiền) vì tiền mua không bị trừ.

In [28]:
def max_future_value(present, future, budget):
    f = [0] * (budget + 1)
    for a, b in zip(present, future):
        for j in range(budget, a - 1, -1):
            f[j] = max(f[j], f[j - a] + b)
    return f[budget]
print(max_future_value(*examples[2][:3]), "(ví dụ 3: mua cổ phiếu 0 và 1, bán được 0 + 3)")

3 (ví dụ 3: mua cổ phiếu 0 và 1, bán được 0 + 3)


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tập con | vét cạn | O(2ⁿ·n) | O(1) | kiểm chứng |
| 1b. Đệ quy có nhớ | top-down | O(n·B) | O(n·B) | dễ nghĩ ra, chậm |
| 2. DP 2 chiều | bảng đầy đủ | O(n·B) | O(n·B) | cần truy vết tập |
| 3. DP 1 chiều ngược | nén | O(n·B) | O(B) | **mặc định** |
| 4. List comprehension | hàng mới từ hàng cũ | O(n·B) | O(B) | nhanh nhất trong Python |
| 5. Theo lợi nhuận | đổi chiều | O(n·P) | O(P) | ngân sách lớn, lời nhỏ |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumProfit([1, 2, 3], [2, 4, 6], 3)` và `maximumProfit([2, 2, 2], [3, 3, 3], 5)`?

**Bài 2 (tính tay):** Trace DP 1 chiều cho `present = [3, 4, 2]`, `future = [5, 8, 3]`, `budget = 6`.

**Bài 3 (điền TODO):** Viết `bt3_todo(present, future, budget)` trả về **mảng `f` đầy đủ** (độ dài `budget + 1`), tức đáp án cho mọi ngân sách `0..budget`.

**Bài 4 (tự viết lại):** Viết `bt4_todo(present, future, budget)` cài bản nộp không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(present, future, budget)` trả về `(lợi nhuận lớn nhất, tổng tiền mua nhỏ nhất để đạt lợi nhuận đó)`.

In [29]:
def bt3_todo(present, future, budget):
    # TODO: mảng f cho mọi ngân sách
    pass

def bt4_todo(present, future, budget):
    # TODO: bản nộp
    pass

def bt5_todo(present, future, budget):
    # TODO: (lời lớn nhất, chi phí nhỏ nhất đạt lời đó)
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: lời lần lượt `1, 2, 3`. Mức 2: ngân sách 3: `{0, 1}` (giá 3, lời 3) hoặc `{2}` (giá 3, lời 3). Mức 3: `3`; câu hai: mua 2 mã giá 2 ⇒ lời `2`.

**Bài 2.** Mức 1: lời `2, 4, 1`. Mức 2: sau cổ phiếu 0: `f = [0,0,0,2,2,2,2]`; sau cổ phiếu 1: `[0,0,0,2,4,4,4]`. Mức 3: sau cổ phiếu 2: `f[6] = max(4, f[4] + 1) = 5` ⇒ `[0,0,1,2,4,4,5]`, đáp án `5`.

**Bài 3.** Mức 1: bản nộp. Mức 2: trả về `f` thay vì `f[budget]`. Mức 3: `f` không giảm.

**Bài 4.** Mức 1: `f = [0] * (budget + 1)`. Mức 2: bỏ cổ phiếu không lời. Mức 3: duyệt `j` ngược.

**Bài 5.** Mức 1: đáp án `P = f[budget]`. Mức 2: chi phí nhỏ nhất = `j` nhỏ nhất mà `f[j] = P` (vì `f` là "tối đa `j`", không giảm). Mức 3: `min(j for j in range(budget + 1) if f[j] == P)`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `3`, `2`. **Bài 2:** `f = [0,0,1,2,4,4,5]`, đáp án `5`.

In [30]:
assert Solution().maximumProfit([1, 2, 3], [2, 4, 6], 3) == 3
assert Solution().maximumProfit([2, 2, 2], [3, 3, 3], 5) == 2

def bt3(present, future, budget):
    f = [0] * (budget + 1)
    for a, b in zip(present, future):
        if b <= a: continue
        for j in range(budget, a - 1, -1):
            f[j] = max(f[j], f[j - a] + b - a)
    return f

assert bt3([3, 4, 2], [5, 8, 3], 6) == [0, 0, 1, 2, 4, 4, 5]

def bt4(present, future, budget):
    return bt3(present, future, budget)[budget]

def bt5(present, future, budget):
    f = bt3(present, future, budget); P = f[budget]
    return P, min(j for j in range(budget + 1) if f[j] == P)

def bt5_brute(present, future, budget):
    n = len(present); best = (0, 0)
    for mask in range(1 << n):
        c = sum(present[i] for i in range(n) if mask >> i & 1)
        g = sum(future[i] - present[i] for i in range(n) if mask >> i & 1)
        if c <= budget and (g > best[0] or (g == best[0] and c < best[1])): best = (g, c)
    return best

rng = random.Random(1988)
for _ in range(400):
    n = rng.randint(1, 9); p = [rng.randint(0, 10) for _ in range(n)]; f = [rng.randint(0, 14) for _ in range(n)]
    b = rng.randint(0, 35)
    assert bt3(p, f, b) == [brute(p, f, x) for x in range(b + 1)]
    assert bt4(p, f, b) == brute(p, f, b)
    assert bt5(p, f, b) == bt5_brute(p, f, b)
print(bt5(*examples[0][:3]), bt5(*examples[2][:3]))
print("Đáp án bài tập: tất cả assert OK")

(6, 10) (0, 0)
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f = [0] * (budget + 1)                     # "ngân sách tối đa j"
for a, b in zip(present, future):
    if b <= a: continue                    # không lời: bỏ
    for j in range(budget, a - 1, -1):     # 0/1 => duyệt NGƯỢC
        f[j] = max(f[j], f[j - a] + b - a)
return f[budget]
```

**Checklist:**
- [ ] Trọng lượng = giá hiện tại, giá trị = `future − present`.
- [ ] Duyệt ngân sách **ngược** (mỗi mã mua tối đa một lần).
- [ ] Khởi tạo 0 ⇒ trả `f[budget]`; khởi tạo `−∞` ⇒ trả `max(f)`.
- [ ] Không tham lam theo tỉ lệ / theo lời lớn nhất.
- [ ] Cổ phiếu giá 0 vẫn xử lý đúng (`range(budget, -1, -1)`).
- [ ] (Premium) Đối chiếu lại đề gốc.